In [15]:
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson
import statsmodels.api as sm
import numpy as np
import pandas as pd
from statsmodels.stats.sandwich_covariance import cov_hac #heteroscedasticity and autocorrelation robust covariance matrix (Newey-West)
from statsmodels.regression.linear_model import OLS

In [16]:
"""IMPORT DES DATAFRAMES DETAIL : 
df_best_param : pas utilité
df_in : prédictions in-sample
df_oos : prédictions out-of-sample 
df_r2_split_oos : r2 de chaque split """

df_best_param = pd.read_excel("df_best_param.xlsx")
df_in = pd.read_excel("df_in.xlsx")
df_oos = pd.read_excel("df_oos.xlsx")
df_r2_split_oos = pd.read_excel("df_r2_split_oos.xlsx")

MESURES R² OOS IN SAMPLE BENCHMARK

In [17]:
"""FONCTIONS METRIQUES :
R² : calcul du R² selon le papier de Gu et al. (2020) 
% ratio : détermine si le modèle prédit bien le bon signe de la prédiction 
R2 benchmark : MSEmodel / MSEha (Xiu and Liu 2024)
"""
#R²
def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum(y_true**2) 
    return 1 - ss_res/ss_tot if ss_tot != 0 else np.nan

#% ratio:
def success_ratio(y_true, y_pred, ignore_zero=True):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    sign_true = np.sign(y_true)
    sign_pred = np.sign(y_pred)

    if ignore_zero:
        mask = sign_true != 0
        sign_true = sign_true[mask]
        sign_pred = sign_pred[mask]

    if len(sign_true) == 0:
        return np.nan  
    return (sign_true == sign_pred).mean()

#R2 benchmark 
def r2_vs_benchmark(y_true, y_pred_model, y_pred_bench):
    T = len(y_true)
    mspe_model = (1/T) * np.sum((y_true - y_pred_model)**2)
    mspe_bench = (1/T) * np.sum((y_true - y_pred_bench)**2)
    return 1 - (mspe_model / mspe_bench)

In [18]:
#Tableau in sample contient : R² in sample, Success ratio in sample, comparaison avec HA 

predictions_in = {
    "OLS": df_in["y_trainval_ols"],
    "PLS": df_in["y_trainval_pls"],
    "PCR": df_in["y_trainval_pcr"],
    "Enet": df_in["y_trainval_en"],
    "RF": df_in["y_trainval_rf"],
    "GBRT": df_in["y_trainval_gbrt"],
    "XGB": df_in["y_trainval_xgb"],
    "HA": df_in["y_trainval_pred_ha"]
}

y_trainval_pred_ha = df_in["y_trainval_pred_ha"]
y_trainval_true = df_in["y_trainval_true"]

#TABLE 1 ET 2 : R², success ratio, comparaison HA
rows = []
for model_name, y_pred in predictions_in.items():
    r2_vs = r2_vs_benchmark(y_trainval_true, y_pred, y_trainval_pred_ha)
    r2_in = r2(y_trainval_true, y_pred)
    sr_in = success_ratio(y_trainval_true, y_pred)
    rows.append({
        "Model" : model_name, 
        "In-sample $R^2$": r2_in,
        "Success Ratio": sr_in,
        "$R^2$ vs HA" : r2_vs })
    
df_r2_in_vs = pd.DataFrame(rows)

# Pivot simple pour transformer les splits en colonnes

print(df_r2_in_vs)
#print(df_r2_in_vs)
#print(df_pivot)

##Conversion → Latex 
df_latex_r2in = df_r2_in_vs.copy()
cols_to_convert = ["$R^2$ vs HA", "In-sample $R^2$", "Success Ratio"]

for col in cols_to_convert:
    df_latex_r2in[col] = (df_latex_r2in[col] * 100).apply(lambda x: f"{x:.2f}") #* 100 et arrondir trois chiffres après la virgule

latex_table = df_latex_r2in.to_latex(index=False, escape=False)
print(latex_table)

  Model  In-sample $R^2$  Success Ratio  $R^2$ vs HA
0   OLS         0.037758       0.572194     0.018679
1   PLS         0.030687       0.568668     0.011467
2   PCR         0.028940       0.568821     0.009686
3  Enet         0.031097       0.570584     0.011886
4    RF         0.080749       0.578753     0.062522
5  GBRT         0.076823       0.595312     0.058518
6   XGB         0.073586       0.581939     0.055218
7    HA         0.019442       0.565601     0.000000
\begin{tabular}{llll}
\toprule
Model & In-sample $R^2$ & Success Ratio & $R^2$ vs HA \\
\midrule
OLS & 3.78 & 57.22 & 1.87 \\
PLS & 3.07 & 56.87 & 1.15 \\
PCR & 2.89 & 56.88 & 0.97 \\
Enet & 3.11 & 57.06 & 1.19 \\
RF & 8.07 & 57.88 & 6.25 \\
GBRT & 7.68 & 59.53 & 5.85 \\
XGB & 7.36 & 58.19 & 5.52 \\
HA & 1.94 & 56.56 & 0.00 \\
\bottomrule
\end{tabular}



In [21]:
print(df_oos.columns)

Index(['Unnamed: 0', 'Date', 'Ticker', 'y_true', 'y_pred_ols', 'y_pred_pls',
       'y_pred_pcr', 'y_pred_en', 'y_pred_rf', 'y_pred_gbrt', 'y_pred_xgb',
       'y_pred_ha'],
      dtype='object')


In [22]:
#Calculs métriques out-of-sample

#Prédictions out of sample 
y_true = df_oos["y_true"]

predictions_oos = {
    "OLS": df_oos["y_pred_ols"],
    "PLS": df_oos["y_pred_pls"],
    "PCR": df_oos["y_pred_pcr"],
    "Enet": df_oos["y_pred_en"],
    "RF": df_oos["y_pred_rf"],
    "GBRT": df_oos["y_pred_gbrt"],
    "XGB": df_oos["y_pred_xgb"],
    "HA": df_oos["y_pred_ha"]
}

#Calcul de R²
rows = []
for model_name, y_pred in predictions_oos.items():
    r2_oos = r2(y_true, y_pred)
    rows.append({
        "Model" : model_name, 
        "Out-of-sample $R^2$": r2_oos,})
df_r2_oos = pd.DataFrame(rows)

#R² par split 


print(df_r2_oos.head(10))

  Model  Out-of-sample $R^2$
0   OLS             0.024993
1   PLS             0.023963
2   PCR             0.022647
3  Enet             0.026295
4    RF             0.026319
5  GBRT             0.016348
6   XGB             0.033540
7    HA             0.012189


In [19]:

models = [col for col in df_oos.columns if col.startswith('y_pred_')]
results = []

for i in range(len(models)):
    for j in range(i+1, len(models)):
        model1 = models[i]
        model2 = models[j]

        #calcule mse du modèle 1 et 2 
        e1 = (df_oos['y_true'] - df_oos[model1])**2
        e2 = (df_oos['y_true'] - df_oos[model2])**2
        d = e1 - e2
        d = d.dropna()

        X = np.ones(len(d))  # régression constante
        model = sm.OLS(d, X).fit()
        cov = cov_hac(model, nlags=1)  
        se = np.sqrt(cov[0][0])
        dm_stat = d.mean() / se
        p_value = 2 * (1 - t.cdf(abs(dm_stat), df=len(d) - 1))

        results.append({
            'Model 1': model1,
            'Model 2': model2,
            'DM Stat': dm_stat,
            'P-Value': p_value,
            'Best Model': model2 if dm_stat > 0 else model1
        })
  
dm_df = pd.DataFrame(results)
print(dm_df)

NameError: name 't' is not defined

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd

# (Optionnel) Style cohérent avec LaTeX
matplotlib.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Palatino"],
    "figure.figsize": (6.2, 4.2),  # ≈ \textwidth, pour Overleaf
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9
})

# Exemple : df_best_param = pd.read_csv("best_params.csv")
# Ton dataframe doit contenir les colonnes : 'Date', 'Best lambda', 'Best_k_pcr', ...

fig, axes = plt.subplots(2, 3, figsize=(6.2, 4), sharex=True)
fig.suptitle(r"\textbf{Time-varying Model Complexity}", fontsize=11)

model_params = [
    (r"ENet+H", "Best lambda", r"\# of Char."),
    (r"PCR", "Best_k_pcr", r"\# of Comp."),
    (r"PLS", "Best_k_pls", r"\# of Comp."),
    (r"RF", "max_depths_rf", "Tree Depth"),
    (r"GBRT+H", "max_depths_gbrt", r"\# of Char."),
    (r"XGB", "max_depths_xgb", "Tree Depth")
]

for ax, (title, col, ylabel) in zip(axes.flatten(), model_params):
    ax.plot(df_best_param["Date"], df_best_param[col])
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Year")
    ax.grid(True)

plt.tight_layout(rect=[0, 0, 1, 0.95])

# Export PDF (vectoriel, qualité Overleaf)
plt.savefig("model_complexity.pdf", bbox_inches="tight")


RuntimeError: 'pdflatex' not found; install it or change rcParams['pgf.texsystem'] to an available TeX implementation

Error in callback <function _draw_all_if_interactive at 0x00000171121CCEA0> (for post_execute), with arguments args (),kwargs {}:


RuntimeError: 'pdflatex' not found; install it or change rcParams['pgf.texsystem'] to an available TeX implementation

PORTEFEUILLES

In [ ]:
df_me = pd.read_excel("me_df.xlsx")
df_me = df_me.sort_values("Date").reset_index(drop=True) 

In [ ]:
df_predict = df_oos.copy()

In [ ]:
#I. PORTEFEUILLES EQUALLY WEIGHT

pred_cols = ['y_pred_ols','y_pred_pls','y_pred_pcr','y_pred_en',
             'y_pred_rf','y_pred_gbrt','y_pred_xgb','y_pred_ha']

col_ret = 'y_true'
nb_groups = 3  

# Résultats finaux
final_table = {}

for col_pred in pred_cols:
    all_rows = []

    for date, i in df_predict.groupby('Date'):
        i = i.sort_values(col_pred).reset_index(drop=True)
        n = len(i)
        size = n // nb_groups
        groups = [min(idx // size, nb_groups - 1) for idx in range(n)]
        i["group"] = groups

        for grp in range(nb_groups):
            sub = i[i["group"] == grp]
            mean_pred = sub[col_pred].mean()
            mean_real = sub[col_ret].mean()
            std_real = sub[col_ret].std()
            sr_real = mean_real / std_real if std_real != 0 else np.nan

            all_rows.append({
                "Group": grp,
                "Pred": mean_pred,
                "Avg": mean_real,
                "SD": std_real,
                "SR": sr_real
            })

    # Convertir en DataFrame
    df_result = pd.DataFrame(all_rows)
    df_grouped = df_result.groupby("Group").mean().reset_index()

    # Ajouter H-L
    hl_row = {
        "Group": "H-L",
        "Pred": df_grouped.loc[nb_groups-1, "Pred"] - df_grouped.loc[0, "Pred"],
        "Avg":  df_grouped.loc[nb_groups-1, "Avg"]  - df_grouped.loc[0, "Avg"],
        "SD":   df_grouped.loc[nb_groups-1, "SD"],  # ou recompute spread SD
        "SR":   (df_grouped.loc[nb_groups-1, "Avg"] - df_grouped.loc[0, "Avg"]) / df_grouped["SD"].mean()
    }

    df_grouped = pd.concat([df_grouped, pd.DataFrame([hl_row])], ignore_index=True)
    final_table[col_pred.upper()] = df_grouped

# 🔁 Tu peux afficher tous les tableaux :
for model, df in final_table.items():
    print(f"\n==== {model} ====")
    print(df.to_string(index=False))


==== Y_PRED_OLS ====
Group      Pred      Avg       SD       SR
    0 -0.005306 0.000054 0.059028 0.025086
    1  0.011263 0.011722 0.054833 0.238154
    2  0.028940 0.019790 0.058492 0.313894
  H-L  0.034245 0.019735 0.058492 0.343518

==== Y_PRED_PLS ====
Group     Pred      Avg      SD       SR
    0 0.001327 0.004931 0.05307 0.101361
    1 0.011404 0.011917 0.05253 0.249925
    2 0.022696 0.015139 0.06440 0.238262
  H-L 0.021369 0.010208 0.06440 0.180134

==== Y_PRED_PCR ====
Group     Pred      Avg       SD       SR
    0 0.002819 0.005899 0.051606 0.121845
    1 0.011550 0.011110 0.052756 0.255101
    2 0.021166 0.014975 0.065886 0.226640
  H-L 0.018346 0.009076 0.065886 0.159928

==== Y_PRED_EN ====
Group     Pred      Avg       SD       SR
    0 0.001139 0.000696 0.057801 0.034711
    1 0.011619 0.011736 0.055621 0.249045
    2 0.022654 0.019178 0.058756 0.302749
  H-L 0.021515 0.018482 0.058756 0.322026

==== Y_PRED_RF ====
Group     Pred      Avg       SD       SR
    0 0.00

In [ ]:
# Associe la market cap à df_predict
#df_predict = df_predict.merge(df_me[['Date', 'Ticker', 'Mkt_Cap_Monthly']], on=['Date', 'Ticker'], how='left')

In [ ]:


col_ret = 'y_true'
col_weight = 'Mkt_Cap_Monthly'
nb_groups = 3

final_table_vw = {}

for col_pred in pred_cols:
    all_rows = []

    for date, i in df_predict.groupby('Date'):
        i = i.sort_values(col_pred).reset_index(drop=True)
        i[col_weight] = i[col_weight].fillna(0)

        n = len(i)
        size = n // nb_groups
        groups = [min(idx // size, nb_groups - 1) for idx in range(n)]
        i["group"] = groups

        for grp in sorted(i["group"].unique()):
            sub = i[i["group"] == grp]
            w = sub[col_weight]
            if w.sum() == 0:
                continue  # éviter division par zéro

            w_norm = w / w.sum()
            pred_mean = np.average(sub[col_pred], weights=w_norm)
            ret_mean = np.average(sub[col_ret], weights=w_norm)
            ret_std = np.sqrt(np.average((sub[col_ret] - ret_mean)**2, weights=w_norm))
            ret_sr = ret_mean / ret_std if ret_std != 0 else np.nan

            all_rows.append({
                "Group": grp,
                "Pred": pred_mean,
                "Avg": ret_mean,
                "SD": ret_std,
                "SR": ret_sr
            })

    df_result = pd.DataFrame(all_rows)
    df_grouped = df_result.groupby("Group").mean().reset_index()

    # Ajouter ligne H-L
    hl_row = {
        "Group": "H-L",
        "Pred": df_grouped.loc[nb_groups-1, "Pred"] - df_grouped.loc[0, "Pred"],
        "Avg": df_grouped.loc[nb_groups-1, "Avg"] - df_grouped.loc[0, "Avg"],
        "SD": df_grouped.loc[nb_groups-1, "SD"],  # facultatif : recompute spread SD
        "SR": (df_grouped.loc[nb_groups-1, "Avg"] - df_grouped.loc[0, "Avg"]) / df_grouped["SD"].mean()
    }

    df_grouped = pd.concat([df_grouped, pd.DataFrame([hl_row])], ignore_index=True)
    final_table_vw[col_pred.upper()] = df_grouped

# 🔁 Affichage
for model, df in final_table_vw.items():
    print(f"\n==== {model} (VW) ====")
    print(df.to_string(index=False))


KeyError: 'Mkt_Cap_Monthly'